In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#building DINO ops
import torch
import os

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

!git clone https://github.com/IDEA-Research/DINO.git
%cd DINO

!pip install -q -r requirements.txt
!pip install -q opencv-python matplotlib scipy

cuda_file_path = 'models/dino/ops/src/cuda/ms_deform_attn_cuda.cu'

with open(cuda_file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'AT_DISPATCH_FLOATING_TYPES(value.type()',
    'AT_DISPATCH_FLOATING_TYPES(value.scalar_type()'
)

with open(cuda_file_path, 'w') as f:
    f.write(content)

%cd models/dino/ops
!python setup.py build install
!python test.py

In [ ]:
#Building MultiScaleDeformableAttention
import os
import sys
import glob
import shutil

if os.path.exists('/content/DINO/DINO'):
    ops_path = '/content/DINO/DINO/models/dino/ops'
else:
    ops_path = '/content/DINO/models/dino/ops'

%cd {ops_path}

!rm -rf build/ dist/ *.egg-info *.so
!python setup.py build_ext --inplace

so_files = glob.glob('*.so')
if not so_files:
    build_so = glob.glob('build/**/*.so', recursive=True)
    for so in build_so:
        shutil.copy(so, '.')

if ops_path not in sys.path:
    sys.path.insert(0, ops_path)

if os.path.exists('/content/DINO/DINO'):
    %cd /content/DINO/DINO
else:
    %cd /content/DINO

try:
    import MultiScaleDeformableAttention as MSDA
    print(f"Success: {MSDA.__file__}")
except ImportError as e:
    print(f"Failed: {e}")
    !ls -la {ops_path}/*.so

In [ ]:
%cd DINO/



In [ ]:
import os, sys
import torch, json
import numpy as np
from main import build_model_main
from util.slconfig import SLConfig
from datasets import build_dataset
from util.visualizer import COCOVisualizer
from util import box_ops

In [ ]:
model_config_path = "/content/DINO/config/DINO/DINO_5scale.py"
model_checkpoint_path = "/content/drive/MyDrive/DINO_Thesis_Optimized/checkpoint0044.pth"

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1
%env PYTORCH_ALLOC_CONF=expandable_segments:True

!python main.py \
    -c config/DINO/DINO_5scale.py \
    --dataset_file coco \
    --coco_path "/content/drive/MyDrive/FullFrameDatasetForParticle Filter/DJI_0004/COCODIR" \
    --output_dir "/content/drive/MyDrive/FullFrameDatasetForParticle Filter" \
    --pretrain_model_path "/content/checkpoint0031_5scale.pth" \
    --finetune_ignore label_enc.weight class_embed \
    --amp \
    --num_workers 2 \
    --options \
        modelname=dino \
        backbone=resnet50 \
        num_classes=6 \
        dn_labelbook_size=7 \
        lr=0.0001 \
        lr_backbone=1e-05 \
        lr_drop=11 \
        batch_size=1 \
        epochs=12 \
        weight_decay=0.0001 \
        clip_max_norm=0.1 \
        onecyclelr=False \
        multi_step_lr=False \
        lr_drop_list=[11] \
        save_checkpoint_interval=1 \
        dn_scalar=50 \
        embed_init_tgt=TRUE \
        dn_label_coef=1.0 \
        dn_bbox_coef=1.0 \
        use_ema=False \
        dn_box_noise_scale=1.0

In [ ]:
# RESTART RUNTIME if you had a crash
%env PYTORCH_ALLOC_CONF=expandable_segments:True

!python main.py \
    -c config/DINO/DINO_5scale.py \
    --dataset_file coco \
    --coco_path "/content/drive/MyDrive/FullFrameDatasetForParticle Filter/DJI_0004/COCODIR" \
    --output_dir "/content/drive/MyDrive/DINO_Thesis_Optimized" \
    --resume "/content/drive/MyDrive/FullFrameDatasetForParticle Filter/checkpoint0011.pth" \
    --amp \
    --num_workers 2 \
    --options \
        modelname=dino \
        backbone=resnet50 \
        num_classes=6 \
        dn_labelbook_size=7 \
        lr=5e-05 \
        lr_backbone=1e-05 \
        lr_drop=40 \
        batch_size=1 \
        epochs=50 \
        onecyclelr=False \
        multi_step_lr=False \
        lr_drop_list=[40] \
        weight_decay=0.0001 \
        clip_max_norm=0.1 \
        two_stage_default_hw=0.01 \
        set_cost_giou=5.0 \
        bbox_loss_coef=10.0 \
        giou_loss_coef=4.0 \
        dn_scalar=100 \
        dn_box_noise_scale=1.0 \
        dn_label_coef=1.0 \
        dn_bbox_coef=1.0 \
        save_checkpoint_interval=1 \
        use_ema=False

In [ ]:
# RESTART RUNTIME to completely clear the optimizer's memory
%env PYTORCH_ALLOC_CONF=expandable_segments:True

!python main.py \
    -c config/DINO/DINO_5scale.py \
    --dataset_file coco \
    --coco_path "/content/drive/MyDrive/FullFrameDatasetForParticle Filter/DJI_0004/COCODIR" \
    --output_dir "/content/drive/MyDrive/DINO_Thesis_Final_Run" \
    --pretrain_model_path "/content/checkpoint0031_5scale.pth" \
    --finetune_ignore label_enc.weight class_embed \
    --amp \
    --num_workers 2 \
    --options \
        modelname=dino \
        backbone=resnet50 \
        num_classes=6 \
        dn_labelbook_size=7 \
        lr=5e-05 \
        lr_backbone=1e-05 \
        lr_drop=25 \
        batch_size=1 \
        epochs=40 \
        onecyclelr=False \
        multi_step_lr=False \
        lr_drop_list=[25] \
        weight_decay=0.0001 \
        clip_max_norm=0.1 \
        two_stage_default_hw=0.01 \
        set_cost_giou=5.0 \
        bbox_loss_coef=10.0 \
        giou_loss_coef=4.0 \
        dn_scalar=100 \
        dn_box_noise_scale=1.0 \
        dn_label_coef=1.0 \
        dn_bbox_coef=1.0 \
        save_checkpoint_interval=1 \
        use_ema=False

In [ ]:
import fileinput

# This surgically adds 'weights_only=False' to every torch.load call in main.py
for line in fileinput.input('/content/DINO/main.py', inplace=True):
    print(line.replace("torch.load(", "torch.load(weights_only=False, "), end='')

print("✅ main.py patched for PyTorch 2.6 compatibility!")